## CLASS 46 — __str__ & __repr__


### 👨‍🏫 Concept 1 — problem: default print bekaar hai


In [3]:
class Dog:
    def __init__(self, name):
        self.name = name

d = Dog("Tommy")
print(d)        # <__main__.Dog object at 0x000001A2...>  😖 useless

### 👨‍🏫 Concept 2 — `__str__` (insaano ke liye sundar)

> **📖 Technical definition — `__str__` and `__repr__`:** These are special ("dunder") methods that define an object's text form. `__str__` returns a readable string for end users (used by `print()` and `str()`), while `__repr__` returns an unambiguous, developer-oriented string for debugging (used by `repr()` and in containers). If only one is defined, `__repr__` acts as the fallback.


In [4]:
class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        return f"{self.name} ({self.age} years old)"

d = Dog("Tommy", 3)
print(d)            # Tommy (3 years old)   ✅ ab sundar hai!

Tommy (3 years old)


### 👨‍🏫 Concept 3 — `__repr__` (developer/debug ke liye)


In [5]:
class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        return f"{self.name}, age {self.age}"

    def __repr__(self):
        return f"Dog(name='{self.name}', age={self.age})"

d = Dog("Tommy", 3)
print(d)            # Tommy, age 3              (__str__ — print ke liye)
print(repr(d))      # Dog(name='Tommy', age=3)  (__repr__ — debug ke liye)
dogs = [Dog("A", 1), Dog("B", 2)]
print(dogs)         # [Dog(name='A', age=1), Dog(name='B', age=2)]  (list __repr__ use karti hai)

Tommy, age 3
Dog(name='Tommy', age=3)
[Dog(name='A', age=1), Dog(name='B', age=2)]


### 💻 Demo — Message object (agent-flavoured)


In [6]:
class Message:
    def __init__(self, role, content):
        self.role = role
        self.content = content

    def __repr__(self):
        return f"Message(role='{self.role}', content='{self.content}')"

msg = Message("user", "Hello AI")
print(msg)          # Message(role='user', content='Hello AI')

history = [Message("user", "Hi"), Message("assistant", "Hello!")]
print(history)      # [Message(...), Message(...)]

Message(role='user', content='Hello AI')
[Message(role='user', content='Hi'), Message(role='assistant', content='Hello!')]


### ❌ Common mistakes


In [7]:
class Dog:
    def __str__(self):
        print(self.name)        # ❌ print mat karo — RETURN karo string
        # sahi: return self.name

class Dog:
    def __str__(self):
        return self.age         # ❌ age int hai — __str__ ko STRING return karna hai
        # sahi: return str(self.age)

## CLASS 47 — More Dunders & Attributes


### 👨‍🏫 Concept 1 — problem: objects compare nahi hote


In [8]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

p1 = Point(1, 2)
p2 = Point(1, 2)
print(p1 == p2)     # False  😖 — same values, par Python ko alag lagte hain

False


### 👨‍🏫 Concept 2 — `__eq__` (apni equality define karo)


> **📖 Technical definition — `__eq__` and `__len__`:** Dunder methods let objects work with built-in operations. `__eq__` defines how the `==` operator compares two objects (usually by comparing their data). `__len__` defines what the built-in `len()` returns for an object.


In [9]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __eq__(self, other):
        return self.x == other.x and self.y == other.y

p1 = Point(1, 2)
p2 = Point(1, 2)
p3 = Point(5, 5)
print(p1 == p2)     # True   ✅ same values = equal
print(p1 == p3)     # False

True
False


### 👨‍🏫 Concept 3 — `__len__` (object ko size do)


In [10]:
class Playlist:
    def __init__(self):
        self.songs = []

    def add(self, song):
        self.songs.append(song)

    def __len__(self):
        return len(self.songs)

p = Playlist()
p.add("Song A")
p.add("Song B")
print(len(p))       # 2   ✅ len() ab hamare object par chalta hai

2


### 👨‍🏫 Concept 4 — class attribute vs instance attribute

> **📖 Technical definition — Class attribute vs instance attribute:** A class attribute is defined on the class itself and shared by all instances. An instance attribute is defined per object (typically in `__init__` via `self`) and holds data unique to that object.


In [11]:
class Dog:
    species = "Canis familiaris"    # CLASS attribute — sab dogs ke liye same

    def __init__(self, name):
        self.name = name            # INSTANCE attribute — har dog ka apna

d1 = Dog("Tommy")
d2 = Dog("Bruno")
print(d1.name, d2.name)         # Tommy Bruno     (alag-alag)
print(d1.species, d2.species)   # dono: Canis familiaris  (shared)

Tommy Bruno
Canis familiaris Canis familiaris


### 👨‍🏫 Concept 5 — class attribute se counter (common pattern)


In [12]:
class User:
    count = 0                       # class attribute — total users

    def __init__(self, name):
        self.name = name
        User.count += 1             # har naye user par badhao

User("Asha")
User("Rahul")
User("Priya")
print(User.count)       # 3   — kitne users bane

3


### ❌ Common mistakes


In [13]:
class Box:
    items = []          # ❌ KHATRA: mutable class attribute sab objects mein share hoga!
# do alag boxes ek hi list share kar lenge (Week 5 wala mutable bug, OOP version)
# sahi: self.items = [] __init__ ke andar (har object ki apni list)

## CLASS 48 — classmethod & staticmethod


### 👨‍🏫 Concept 1 — `@classmethod` (alternative constructor)

> **📖 Technical definition — `@classmethod`:** A class method receives the class itself as its first argument (conventionally `cls`) instead of an instance. It is commonly used to build alternative constructors that create and return a new instance from different input formats.


In [14]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_string(cls, text):
        name, age = text.split(",")     # "Asha,17" ko todo
        return cls(name, int(age))      # naya Person banao

# normal tareeka
p1 = Person("Asha", 17)
# alternative tareeka — string se
p2 = Person.from_string("Rahul,18")
print(p2.name, p2.age)      # Rahul 18

Rahul 18


### 👨‍🏫 Concept 2 — `from_dict` (super common in AI)


In [15]:
class Message:
    def __init__(self, role, content):
        self.role = role
        self.content = content

    @classmethod
    def from_dict(cls, data):
        return cls(data["role"], data["content"])

    def __repr__(self):
        return f"Message('{self.role}', '{self.content}')"

# API se aaya dict
raw = {"role": "user", "content": "Hello"}
msg = Message.from_dict(raw)
print(msg)      # Message('user', 'Hello')

Message('user', 'Hello')


### 👨‍🏫 Concept 3 — `@staticmethod` (utility, object ka data nahi chahiye)


> **📖 Technical definition — `@staticmethod`:** A static method receives neither the instance (`self`) nor the class (`cls`). It is an ordinary function grouped inside a class for logical organisation, called on the class without needing an object.


In [16]:
class MathHelper:
    @staticmethod
    def is_even(n):
        return n % 2 == 0

    @staticmethod
    def add(a, b):
        return a + b

# object banaye bina seedhe call kar sakte ho
print(MathHelper.is_even(10))   # True
print(MathHelper.add(3, 4))     # 7

True
7


### 👨‍🏫 Concept 4 — kaunsa kab? (saaf rule)


| Method type | Pehla parameter | Kab use? |
|---|---|---|
| Normal method | `self` | Object ka data chahiye |
| `@classmethod` | `cls` | Naya object banane ka alag tareeka |
| `@staticmethod` | (kuch nahi) | Bas ek related helper, koi data nahi |


### ❌ Common mistakes


In [17]:
class A:
    @classmethod
    def make(self):         # ❌ classmethod mein 'self' nahi, 'cls' likho
        ...

class A:
    @staticmethod
    def helper(self):       # ❌ staticmethod mein 'self' nahi hona chahiye
        ...

## CLASS 49 — property


### 👨‍🏫 Concept 1 — computed property (apne aap calculate)

> **📖 Technical definition — `@property`:** The `@property` decorator turns a method into a read-only attribute that is accessed without parentheses and computed each time it is read. Adding a matching `@<name>.setter` allows the value to be assigned while running validation logic, combining a simple attribute interface with controlled access.


In [18]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):                 # method, par property ban gaya
        return 3.14159 * self.radius ** 2

c = Circle(5)
print(c.area)       # 78.53975    — dhyaan: c.area() NAHI, bas c.area
c.radius = 10       # radius badlo
print(c.area)       # 314.159     — area apne aap update ho gaya!

78.53975
314.159


### 👨‍🏫 Concept 2 — property with validation (setter)


In [19]:
class Temperature:
    def __init__(self, celsius=0):
        self._celsius = celsius

    @property
    def celsius(self):              # getter — value padhne par
        return self._celsius

    @celsius.setter
    def celsius(self, value):       # setter — value set karne par check
        if value < -273.15:
            raise ValueError("Below absolute zero!")
        self._celsius = value

t = Temperature()
t.celsius = 25          # setter chalta hai (validation pass)
print(t.celsius)        # 25      (getter chalta hai)
t.celsius = -300        # ❌ ValueError: Below absolute zero!

25


ValueError: Below absolute zero!

### 👨‍🏫 Concept 3 — computed property jod kar


In [ ]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    @property
    def area(self):
        return self.width * self.height

    @property
    def perimeter(self):
        return 2 * (self.width + self.height)

r = Rectangle(4, 6)
print(r.area)           # 24
print(r.perimeter)      # 20

### 💻 Demo — Temperature with both units


In [ ]:
class Temp:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def fahrenheit(self):
        return (self.celsius * 9 / 5) + 32

t = Temp(37)
print(t.fahrenheit)     # 98.6
t.celsius = 0
print(t.fahrenheit)     # 32.0

### ❌ Common mistakes


In [ ]:
c = Circle(5)
print(c.area())     # ❌ TypeError — property hai, brackets mat lagao: c.area

# getter aur setter ke naam same hone chahiye
class A:
    @property
    def x(self): return self._x
    @x.setter
    def value(self, v):    # ❌ naam 'x' hona chahiye, 'value' nahi
        self._x = v

## CLASS 50 — Composition (Project Class)


### 👨‍🏫 Concept 1 — composition = object ke andar object

> **📖 Technical definition — Composition:** Composition is a design approach where an object is built from other objects that it holds as attributes (a "has-a" relationship), rather than inheriting from them (an "is-a" relationship). It models systems as collaborating parts and is often more flexible than inheritance.


In [ ]:
class Engine:
    def __init__(self, horsepower):
        self.horsepower = horsepower
    def start(self):
        return "Engine started"

class Car:
    def __init__(self, brand, horsepower):
        self.brand = brand
        self.engine = Engine(horsepower)    # Car ke ANDAR ek Engine object

    def start(self):
        return f"{self.brand}: {self.engine.start()}"

car = Car("Maruti", 80)
print(car.start())              # Maruti: Engine started
print(car.engine.horsepower)    # 80

### 👨‍🏫 Concept 2 — IS-A vs HAS-A (kab kya?)


| Inheritance (IS-A) | Composition (HAS-A) |
|---|---|
| `Dog` IS-A `Animal` | `Car` HAS-A `Engine` |
| Type ka rishta | Parts ka rishta |
| `class Dog(Animal)` | `self.engine = Engine()` |

### 👨‍🏫 Concept 3 — ek object jo kai objects rakhe (list of objects)


In [ ]:
class Tool:
    def __init__(self, name):
        self.name = name

class Toolbox:
    def __init__(self):
        self.tools = []             # Toolbox HAS-A list of Tools

    def add_tool(self, tool):
        self.tools.append(tool)

    def list_tools(self):
        for tool in self.tools:
            print(f"- {tool.name}")

box = Toolbox()
box.add_tool(Tool("Hammer"))
box.add_tool(Tool("Screwdriver"))
box.list_tools()
# - Hammer
# - Screwdriver

### 🛠️ Mini Project — Agent with Tools (structural mock!)


In [ ]:
from abc import ABC, abstractmethod


class Tool(ABC):
    """Base class — har tool ka naam, description aur run() hona zaroori."""
    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description

    @abstractmethod
    def run(self, *args):
        ...

    def __repr__(self):
        return f"Tool(name='{self.name}')"


class CalculatorTool(Tool):
    def __init__(self):
        super().__init__("calculator", "Adds two numbers")

    def run(self, a: float, b: float) -> float:
        return a + b


class GreetTool(Tool):
    def __init__(self):
        super().__init__("greet", "Greets a person by name")

    def run(self, name: str) -> str:
        return f"Hello, {name}!"


class Agent:
    """An agent HAS-A list of tools (composition)."""
    def __init__(self, name: str):
        self.name = name
        self.tools = []                 # composition: agent holds tools

    def add_tool(self, tool: Tool):
        self.tools.append(tool)

    def list_tools(self):
        print(f"{self.name}'s tools:")
        for tool in self.tools:
            print(f"  - {tool.name}: {tool.description}")

    def use_tool(self, tool_name: str, *args):
        for tool in self.tools:
            if tool.name == tool_name:      # naam se sahi tool dhoondho
                return tool.run(*args)
        return f"Tool '{tool_name}' not found"


# --- agent banao aur use karo ---
agent = Agent("Jarvis")
agent.add_tool(CalculatorTool())
agent.add_tool(GreetTool())

agent.list_tools()
# Jarvis's tools:
#   - calculator: Adds two numbers
#   - greet: Greets a person by name

print(agent.use_tool("calculator", 5, 3))    # 8
print(agent.use_tool("greet", "Asha"))       # Hello, Asha!
print(agent.use_tool("unknown"))             # Tool 'unknown' not found

### ❌ Common mistakes


In [ ]:
class Agent:
    tools = []          # ❌ class attribute — saare agents tools share kar lenge!
    # sahi: self.tools = [] __init__ ke andar (har agent ka apna)

agent.use_tool("calculator", 5)   # ❌ CalculatorTool.run ko 2 numbers chahiye